In [ ]:
import pandas as pd
import json

# Load all CSVs
roads = pd.read_csv("../data/road_network.csv")
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
weather = pd.read_csv("../data/weather_conditions.csv")
events = pd.read_csv("../data/traffic_events.csv")

# Load JSON
with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

# Convert sensors JSON into DataFrame
sensors_df = pd.DataFrame(sensors["sensors"])



In [ ]:
# Merge sensors with roads
roads_sensors = sensors_df.merge(roads, left_on="road_segment_id", right_on="road_id", how="left")
print(roads_sensors.head())

# Merge traffic with sensors
traffic_full = traffic.merge(sensors_df, on="sensor_id", how="left")
print(traffic_full.head())

# Merge traffic with weather
traffic_weather = traffic_full.merge(weather, on="timestamp", how="left")

# Check events columns before merging
print(events.columns)

# Example merge (adjust keys if needed)
traffic_all = traffic_weather.merge(
    events,
    left_on="timestamp",
    right_on="timestamp_start",
    how="left"
)

print(traffic_all.head())
print(traffic_all.info())

In [ ]:
print(traffic_all.head())
print(traffic_all.info())


In [ ]:
import pandas as pd

# Road network data
roads = pd.read_csv("../data/road_network.csv")
print("Roads:", roads.shape)
print(roads.head())

# Traffic sensor data
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
print("Traffic:", traffic.shape)
print(traffic.head())

# Weather conditions
weather = pd.read_csv("../data/weather_conditions.csv")
print("Weather:", weather.shape)
print(weather.head())

# Traffic events
events = pd.read_csv("../data/traffic_events.csv")
print("Events:", events.shape)
print(events.head())


In [ ]:
import json

with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

# If it's a dictionary, show keys
print(sensors.keys())

# If you want to peek inside
for key, value in list(sensors.items())[:2]:
    print(key, value)


In [ ]:
traffic.info()


In [ ]:
weather = pd.read_csv("../data/weather_conditions.csv")
print(weather.head())
events = pd.read_csv("../data/traffic_events.csv")
print(events.head())


In [ ]:
traffic.describe()


In [ ]:
traffic.isna().sum()


In [ ]:
traffic_all["timestamp"] = pd.to_datetime(traffic_all["timestamp"])
traffic_all["hour"] = traffic_all["timestamp"].dt.hour
traffic_all["day_of_week"] = traffic_all["timestamp"].dt.dayofweek
traffic_all["is_weekend"] = traffic_all["day_of_week"].isin([5,6]).astype(int)


In [ ]:
import pandas as pd
import json

# -----------------------------
# 1. Load datasets
# -----------------------------
roads = pd.read_csv("../data/road_network.csv")
traffic = pd.read_csv("../data/traffic_sensor_data.csv")
weather = pd.read_csv("../data/weather_conditions.csv")
events = pd.read_csv("../data/traffic_events.csv")

with open("../data/sensor_locations.json") as f:
    sensors = json.load(f)

sensors_df = pd.DataFrame(sensors["sensors"])

# -----------------------------
# 2. Attach road attributes to sensors (map instead of merge)
# -----------------------------
road_map = roads.set_index("road_id").to_dict()

# Add road features directly to sensors_df
for col in roads.columns:
    if col != "road_id":
        sensors_df[col] = sensors_df["road_segment_id"].map(road_map[col])

# -----------------------------
# 3. Attach sensor + road info to traffic (map instead of merge)
# -----------------------------
sensor_map = sensors_df.set_index("sensor_id").to_dict()

for col in sensors_df.columns:
    if col != "sensor_id":
        traffic[col] = traffic["sensor_id"].map(sensor_map[col])

# -----------------------------
# 4. Attach weather info (map by timestamp)
# -----------------------------
weather_map = weather.set_index("timestamp").to_dict()

for col in weather.columns:
    if col != "timestamp":
        traffic[col] = traffic["timestamp"].map(weather_map[col])

# -----------------------------
# 5. Attach event info (map by road_id)
# -----------------------------
events_map = events.set_index("road_id").to_dict()

for col in events.columns:
    if col != "road_id":
        traffic[col] = traffic["road_id"].map(events_map[col])

# -----------------------------
# 6. Feature Engineering
# -----------------------------
traffic["timestamp"] = pd.to_datetime(traffic["timestamp"])
traffic["hour"] = traffic["timestamp"].dt.hour
traffic["day_of_week"] = traffic["timestamp"].dt.dayofweek
traffic["is_weekend"] = traffic["day_of_week"].isin([5,6]).astype(int)

traffic["district"] = traffic["district"].astype("category").cat.codes
traffic["surface_condition"] = traffic["surface_condition"].astype("category").cat.codes
traffic["event_type"] = traffic["event_type"].astype("category").cat.codes

traffic["vehicle_count_lag1"] = traffic["vehicle_count"].shift(1)
traffic["vehicle_count_lag2"] = traffic["vehicle_count"].shift(2)

# -----------------------------
# 7. Inspect final dataset
# -----------------------------
print(traffic.head())
print(traffic.info())


In [1]:
import pandas as pd

# Load only 50k rows and only essential columns
traffic = pd.read_csv(
    "../data/traffic_sensor_data.csv",
    usecols=["sensor_id","timestamp","vehicle_count","average_speed_kmh"],
    nrows=50000
)

print(traffic.head())
print(traffic.info())


             timestamp sensor_id  vehicle_count  average_speed_kmh
0  2024-01-01 00:00:00   SEN-001             14               67.8
1  2024-01-01 00:00:00   SEN-002             23               66.8
2  2024-01-01 00:00:00   SEN-003             22               80.0
3  2024-01-01 00:00:00   SEN-004              8               80.0
4  2024-01-01 00:00:00   SEN-005             13               80.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   timestamp          50000 non-null  object 
 1   sensor_id          50000 non-null  object 
 2   vehicle_count      50000 non-null  int64  
 3   average_speed_kmh  50000 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.5+ MB
None


In [2]:
traffic["timestamp"] = pd.to_datetime(traffic["timestamp"])
traffic["hour"] = traffic["timestamp"].dt.hour
traffic["day_of_week"] = traffic["timestamp"].dt.dayofweek
traffic["is_weekend"] = traffic["day_of_week"].isin([5,6]).astype(int)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Features and target
X = traffic[["hour","day_of_week","is_weekend","average_speed_kmh"]]
y = traffic["vehicle_count"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train baseline model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model R^2 score:", model.score(X_test, y_test))


Model R^2 score: 0.5920427163818838


In [5]:
# Ensure both timestamps are datetime
traffic["timestamp"] = pd.to_datetime(traffic["timestamp"])
weather_small["timestamp"] = pd.to_datetime(weather_small["timestamp"])

# Now merge safely
traffic = traffic.merge(weather_small, on="timestamp", how="left")

